In [1]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent.parent))

from data.OT2.ot2 import PUMP
from otp.core.units import BAR, K_TO_C, P_A, g
from otp.propellants.propellant import Propellant
from pyfluids import FluidsList, Input
from otp.experiments.loaders.airborne import load_run
from scipy.optimize import brentq
import numpy as np
import matplotlib.pyplot as plt

T_K = 293.15
P_IN = 10 * BAR

def rho(fl):
    return fl.fluid.with_state(Input.pressure(1 * BAR), Input.temperature(T_K + K_TO_C)).density

WATER = Propellant("Water", FluidsList.Water, t_tank=T_K, p_tank=P_IN)
class IPA:                      # CoolProp has no propan-2-ol; 20 C literature values
    class _s: density, kinematic_viscosity, pressure = 785.4, 2.66e-6, 4400.0
    class fluid:
        with_state = staticmethod(lambda *a: IPA._s)   # .pressure only read for p_sat

from data.OT2.ot2 import CHANNEL_MAP, DATA_PATH
run = load_run(DATA_PATH, cmap=CHANNEL_MAP)
STEADY_STATE_START, STEADY_STATE_END = (5.0, 10.0)

p_orifice_in = run["p_orifice_in"].mean(STEADY_STATE_START, STEADY_STATE_END)
print(f"p_orifice_in = {p_orifice_in:.2f} bar")

# Tests
IPA_MDOT = np.array([0.706, 0.503, 0.301]) #kg/s
IPA_DP = np.array([12.84, 6.59, 2.31]) * BAR #Pa

CdA = IPA_MDOT / np.sqrt(2 * rho(IPA) * IPA_DP)

# Water Mdot
WATER_MDOT = CdA * np.sqrt(2 * rho(WATER) * (p_orifice_in - 1) * BAR)
print(CdA)
print(WATER_MDOT)
print("")
# Print mean of both
print(f"Mean CdA = {CdA.mean():.6g} m^2, Mean WATER_MDOT = {WATER_MDOT.mean():.3f} kg/s")


print(PUMP.point_performance(IPA, 1.7/787, 33000, 10*BAR, T_K, "lock"))

USER_HOME_DIR=C:\Users\Martin
p_orifice_in = 88.45 bar
[1.57203331e-05 1.56338101e-05 1.58015634e-05]
[2.07712033 2.06568809 2.08785326]

Mean CdA = 1.57186e-05 m^2, Mean WATER_MDOT = 2.077 kg/s
PumpPerformance
  ----- Inputs -----
    propellant                 <class '__main__.IPA'> pumped propellant               
    Q                                         0.00216 m^3/s                 2.16 L/s  pump flow rate
    rpm                                       3.3e+04 RPM                             pump rotational speed
    p_upstream                                  1e+06 Pa                      10 bar  pump upstream pressure
    T_upstream                                    293 K                       20 C    pump upstream temperature
  ----- Outputs -----
    omega                                    3.46e+03 rad/s                           pump rotational speed in rad/s
    mdot                                          1.7 kg/s                            pump mass flow rate
    dp

In [11]:
perf = PUMP.point_performance(WATER, 2.0/1000, 29800, 17*BAR, T_K, "lock")
print(perf.section("Outputs"))
print(perf.section("Head and Power"))
print(perf.section("Pressures"))

PumpPerformance / Outputs
  omega              3.12e+03 rad/s             pump rotational speed in rad/s
  mdot                      2 kg/s              pump mass flow rate
  dp                 7.37e+06 Pa      73.7 bar  pump pressure rise
  specific_speed         9.28 -                 pump specific speed
PumpPerformance / Head and Power
  H_total_real             752 m              pump head
  H_static_real            734 m              pump ideal head
  H_loss_diff             21.8 m              pump diffuser head loss
  eta_hydraulic          0.972 -              pump hydraulic efficiency
  P_shaft             2.28e+04 W     22.8 kW  pump shaft power
  P_hydraulic         1.52e+04 W     15.2 kW  pump hydraulic power
  P_disc_friction     7.68e+03 W     7.68 kW  pump disc friction power
  P_useful            1.47e+04 W     14.7 kW  pump useful power
  eta_power              0.645 -              pump power efficiency
  torque                  7.32 N*m            pump torque
PumpPerf

In [13]:
perf = PUMP.point_performance(IPA, 1.1/787, 19000, 10*BAR, T_K, "lock")
print(perf.section("Outputs"))
print(perf.section("Head and Power"))
print(perf.section("Pressures"))

PumpPerformance / Outputs
  omega              1.99e+03 rad/s             pump rotational speed in rad/s
  mdot                    1.1 kg/s              pump mass flow rate
  dp                 2.35e+06 Pa      23.5 bar  pump pressure rise
  specific_speed         9.73 -                 pump specific speed
PumpPerformance / Head and Power
  H_total_real             305 m              pump head
  H_static_real            296 m              pump ideal head
  H_loss_diff             10.7 m              pump diffuser head loss
  eta_hydraulic          0.966 -              pump hydraulic efficiency
  P_shaft             5.48e+03 W     5.48 kW  pump shaft power
  P_hydraulic          3.4e+03 W      3.4 kW  pump hydraulic power
  P_disc_friction     2.08e+03 W     2.08 kW  pump disc friction power
  P_useful            3.29e+03 W     3.29 kW  pump useful power
  eta_power                0.6 -              pump power efficiency
  torque                  2.75 N*m            pump torque
PumpPerf

In [12]:
perf = PUMP.point_performance(IPA, 1.7/787, 33000, 10*BAR, T_K, "lock")
print(perf.section("Outputs"))
print(perf.section("Head and Power"))
print(perf.section("Pressures"))

PumpPerformance / Outputs
  omega              3.46e+03 rad/s             pump rotational speed in rad/s
  mdot                    1.7 kg/s              pump mass flow rate
  dp                 7.11e+06 Pa      71.1 bar  pump pressure rise
  specific_speed         9.16 -                 pump specific speed
PumpPerformance / Head and Power
  H_total_real             923 m              pump head
  H_static_real            902 m              pump ideal head
  H_loss_diff             25.5 m              pump diffuser head loss
  eta_hydraulic          0.973 -              pump hydraulic efficiency
  P_shaft             2.55e+04 W     25.5 kW  pump shaft power
  P_hydraulic         1.58e+04 W     15.8 kW  pump hydraulic power
  P_disc_friction     9.76e+03 W     9.76 kW  pump disc friction power
  P_useful            1.54e+04 W     15.4 kW  pump useful power
  eta_power              0.601 -              pump power efficiency
  torque                  7.39 N*m            pump torque
PumpPerf